# ⚡ Light Tracker — Data Science Quickstart

This notebook demonstrates how to load, analyze, and build predictive models using the continuous, feature-engineered dataset from **Light Tracker**.

### Two-Sensor Grid Reliability Principles:
- **Availability** = $\frac{\text{Confirmed ON}}{\text{Confirmed Observed (ON + OFF)}}$
- **Coverage** = $\frac{\text{Confirmed Observed}}{\text{Total Time}}$
- **UNKNOWN** periods (both sensors offline) are isolated to prevent false outage labeling.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Load local or remote 5-minute continuous dataset
data_path = Path("../../frontend/public/data/dataset-timeseries-5m.json")
if data_path.exists():
    with open(data_path, "r", encoding="utf-8") as f:
        records = json.load(f)
    df = pd.DataFrame(records)
else:
    # Remote fallback from site
    url = "http://localhost:3000/data/dataset-timeseries-5m.json"
    df = pd.read_json(url)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)

print(f"Loaded {len(df):,} observations across 7 days.")
df.head()

## 1. Reliability & Coverage Metrics

In [ ]:
state_counts = df['state'].value_counts()
print("State Frequency Distribution:")
print(state_counts)

total_intervals = len(df)
observed_intervals = (df['state'] != 'UNKNOWN').sum()
on_intervals = (df['state'] == 'ON').sum()

availability = (on_intervals / observed_intervals) * 100
coverage = (observed_intervals / total_intervals) * 100

print(f"\nAvailability (of observed time): {availability:.1f}%")
print(f"Coverage (confirmed observable time): {coverage:.1f}%")

## 2. Diurnal Outage Risk by Hour of Day (WAT)

In [ ]:
hourly_stats = df.groupby('hour')['state'].apply(lambda s: (s == 'OFF').mean() * 100)
print("Outage Probability by Hour (%):")
print(hourly_stats.round(1))

peak_hour = hourly_stats.idxmax()
print(f"\nPeak outage risk hour in this dataset: {peak_hour}:00 WAT ({hourly_stats.max():.1f}% outage probability)")

## 3. Outage Duration Analysis (MTTR)

In [ ]:
# Load events dataset
events_path = Path("../../frontend/public/data/dataset-events.json")
if events_path.exists():
    with open(events_path, "r", encoding="utf-8") as f:
        events_df = pd.DataFrame(json.load(f))
    
    outages = events_df[events_df['state'] == 'OFF']
    print(f"Recorded Outages: {len(outages)}")
    print(f"Mean Outage Duration: {outages['duration_hours'].mean():.2f} hours")
    print(f"Median Outage Duration: {outages['duration_hours'].median():.2f} hours")
    print(f"Longest Outage: {outages['duration_hours'].max():.2f} hours")

## 4. Feature Matrix for Outage Classification

In [ ]:
feature_cols = ['hour', 'day_of_week', 'is_weekend', 'sin_hour', 'cos_hour', 'rolling_avail_6h', 'rolling_avail_24h', 'outage_streak_hours']

# Mask out UNKNOWN rows to ensure ground truth credibility
training_df = df[df['is_unknown'] == 0]
X = training_df[feature_cols]
y = training_df['is_grid_on'].astype(int)

print(f"Training matrix ready: {X.shape[0]} samples with {X.shape[1]} features.")
X.head()